# Behind the scenes: Generating texts
In this notebook we will generate texts using a large language model and store them in a CSV file. More precisely, we generate PhD thesis topics from subdomains of computer science using the [gpt-oss:20b model](https://ollama.com/library/gpt-oss) and the [ollama software](https://ollama.com/) for running LLMs locally.

In [1]:
import random

First, we test the connection to the LLM server.

In [2]:
def prompt(message:str, model="gpt-oss:20b", temperature=0.2):
    """A prompt helper function that sends a message to ScaDS.AI LLM server at 
    ZIH TU Dresden and returns only the text response.
    """
    import os
    import openai
    
    # convert message in the right format if necessary
    if isinstance(message, str):
        message = [{"role": "user", "content": message}]
    
    # setup connection to the LLM
    client = openai.OpenAI(base_url="http://localhost:11434/v1",
                           api_key = "none"
            #base_url="https://chat-ai.academiccloud.de/v1",
            #api_key = os.environ.get('KISSKI_API_KEY')
            #base_url="https://llm.scads.ai/v1",
            #api_key=os.environ.get('SCADSAI_API_KEY')
                           # openai/gpt-oss-120b
    )
    response = client.chat.completions.create(
        model=model,
        messages=message,
        temperature=temperature
    )
    
    # extract answer
    return response.choices[0].message.content

prompt("hello world")

'Hello! 👋 How can I assist you today?'

Next, we define a couple of random first and lastnames. We also define a list of institutes.
These will hint the LLM to generate PhD topics that are relevant to a given institute.

In [3]:
import pandas as pd
import numpy as np
import random

# Define separate lists for first names and last names
first_names = ["Alex", "Jordan", "Morgan", "Taylor", "Avery",
               "Sam", "Casey", "Riley", "Jamie", "Dakota",
               "Bailey", "Cameron", "Sydney", "Robin", "Charlie",
               "Dana", "Kris", "Skyler", "Devon", "Reese"]

last_names = ["Reed", "Smith", "Lee", "Garcia", "Patel",
              "Chen", "O'Hara", "Jain", "Kumar", "Singh",
              "Liu", "Davis", "Clark", "Thomas", "Brooks",
              "Flores", "Rivera", "Adams", "Gonzalez", "Campbell"]

faculties = """Artificial Intelligence
Machine Learning
Computer Vision
Natural Language Processing
Robotics
Algorithms and Data Structures
Computational Complexity
Programming Languages
Software Engineering
Operating Systems
Computer Architecture
Computer Networks
Distributed Systems
Database Systems
Cybersecurity
Human-Computer Interaction
Computer Graphics
Information Retrieval
Quantum Computing
Bioinformatics""".split("\n")
len(faculties), faculties[:10]

(20,
 ['Artificial Intelligence',
  'Machine Learning',
  'Computer Vision',
  'Natural Language Processing',
  'Robotics',
  'Algorithms and Data Structures',
  'Computational Complexity',
  'Programming Languages',
  'Software Engineering',
  'Operating Systems'])

In the following loop we select an institute name randomly and ask the LLM server to generate a PhD topic related to the chosen institute.

In [4]:
research_fields = faculties

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

profiles = []

#for field in faculties:
while len(profiles) < 250:    
    # Random name generation from first and last name
    name = f"{random.choice(first_names)} {random.choice(last_names)}"
    field = random.choice(research_fields)

    topic = prompt(f"""Generate an English PhD thesis topic for someone working at a Computer Science Research Institute in group '{field}'. Reply with only the title of the research project and nothing else.""", temperature=0.9)
    topic = topic.strip().replace("</s>", "").replace("\"", "")
    if ":" in topic and " " not in topic.split(":")[0]:
        topic = topic.split(":")[1]
    if "</thinking>" in topic:
        topic = topic.split("</thinking>")[1]

    print(len(topic), topic)
    
    profiles.append({
        "name": name,
        "research_field": field,
        "topic": topic
    })

95 Runtime Verification of Adaptive Microservice Architectures Under Distributed Failure Scenarios
93 Learning‑Based Resilient Coordination of Heterogeneous Robotic Swarms in Dynamic Environments
112 Unsupervised Spatiotemporal Representation Learning Using Diffusion Models for Real‑Time Video Anomaly Detection
126 Uncertainty‑Aware Reinforcement Learning for Automated Hyperparameter Optimization in Large‑Scale Distributed Machine Learning
72 Fine‑Grained Complexity of Dynamic Subgraph Isomorphism in Sparse Graphs
94 Scalable Graph Neural Networks for Pan‑Genomic Variant Calling with Uncertainty Quantification
104 Parameterized Complexity and Kernelization Limits for the Edge‑Disjoint Paths Problem in Directed Graphs
180 Design, Implementation and Formal Verification of a Hybrid Static‑Dynamic Type System for Safe Interoperability Between Statically Typed and Dynamically Typed Programming Paradigms
103 Reinforcement Learning‑Guided Automatic Generation of Resilient Cloud‑Native Microse

Finally, we store the names, institutes and PhD topics to a csv file.

In [5]:
# Create DataFrame and save CSV
df = pd.DataFrame(profiles)
df.to_csv("phd_topics.csv", index=False)

df.head()

,name,research_field,topic
0,Taylor Reed,Software Engineering,Runtime Verification of Adaptive Microservice ...
1,Riley Jain,Robotics,Learning‑Based Resilient Coordination of Heter...
2,Taylor Adams,Computer Vision,Unsupervised Spatiotemporal Representation Lea...
3,Devon Thomas,Machine Learning,Uncertainty‑Aware Reinforcement Learning for A...
4,Alex Lee,Computational Complexity,Fine‑Grained Complexity of Dynamic Subgraph Is...
